# Workshop: Comparative Framework for Regression Models

Let

$$
\mathcal D_N=\{(x_i,y_i)\}_{i=1}^N, \qquad x_i\in\mathbb R^p, \quad y_i\in\mathbb R,
$$

be a supervised regression sample. The basic reference model is the affine predictor

$$
\widehat f(x)=x^\top\widehat\beta+\widehat\beta_0,
$$

usually fitted by minimizing the empirical squared error. The goal of this workshop is to compare
important extensions of this basic model: regularized linear models, stochastic optimization,
kernel methods, probabilistic regressors, tree methods, and ensembles.

## 1. Short Comparative Table

This table is intentionally brief. The detailed theory and code examples are developed after it.

| Model | Main idea | Objective / fitting principle | Best use case | Main limitation |
|---|---|---|---|---|
| `LinearRegressor` | Global affine fit | Minimize squared error | Simple baseline, interpretable effects | Weak for nonlinear structure |
| `Lasso` | Linear fit with sparse coefficients | Squared error plus $\ell_1$ penalty | Feature selection in high dimension | Unstable with strongly correlated predictors |
| `ElasticNet` | Linear fit with sparse and ridge shrinkage | Squared error plus $\ell_1+\ell_2$ penalty | Correlated high-dimensional features | Needs two regularization choices |
| `SGDRegressor` | Linear model trained incrementally | Stochastic optimization of a chosen loss | Very large or streaming datasets | Sensitive to scaling and learning rate |
| `BayesianRidge` | Linear model with Gaussian prior | Posterior/evidence-based ridge shrinkage | Linear prediction with uncertainty | Costly covariance updates for very large $p$ |
| `KernelRidge` | Ridge regression in RKHS | Squared loss plus RKHS norm penalty | Smooth nonlinear regression, moderate $N$ | Stores and solves with the full kernel matrix |
| Gaussian Process Regressor | Bayesian distribution over functions | Marginal likelihood and posterior prediction | Small data with calibrated uncertainty | Exact training scales cubically in $N$ |
| SVR | Maximum-margin regression | $\epsilon$-insensitive loss plus norm control | Robust kernel regression | Kernel version scales poorly with large $N$ |
| `RandomForestRegressor` | Average many randomized trees | Greedy split reduction plus bagging | Nonlinear tabular data | Large forests can be memory-heavy |
| `GradientBoostingRegressor` / XGBoost | Add trees sequentially | Functional gradient descent / second-order boosting | Strong tabular prediction | Requires careful regularization and tuning |

## 2. Classification Table

Some models fit more than one category. The table assigns each model to the category that best
explains its main modeling role in this workshop.

| Category | Associated models | Reason |
|---|---|---|
| Classical linear models | `LinearRegressor` | Direct affine least-squares regression |
| Regularized linear models | `Lasso`, `ElasticNet`, `BayesianRidge` | Linear model plus deterministic or probabilistic shrinkage |
| Models trained by gradient methods | `SGDRegressor`, `GradientBoostingRegressor`, XGBoost | Parameters or functions are updated by gradient information |
| Kernelized models | `KernelRidge`, SVR, Gaussian Process Regressor | Prediction depends on kernel evaluations $k(x_i,x_j)$ |
| Probabilistic models | `BayesianRidge`, Gaussian Process Regressor | Priors, likelihoods, posterior uncertainty, or marginal likelihood are part of the model |
| Tree-based models | `RandomForestRegressor`, `GradientBoostingRegressor`, XGBoost | The base predictor is a decision tree |
| Ensemble models | `RandomForestRegressor`, `GradientBoostingRegressor`, XGBoost | Predictions combine many fitted base learners |

## 3. Shared Tiny Dataset for the Code Examples

The examples use one small synthetic regression dataset. The target contains a linear component,
noise, and a mild nonlinear perturbation. This makes the comparison meaningful without requiring
external files.

In [1]:
import numpy as np

from sklearn.datasets import make_regression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y_linear = make_regression(
    n_samples=160,
    n_features=6,
    n_informative=4,
    noise=12.0,
    random_state=7,
)
y = y_linear + 15.0 * np.sin(X[:, 0])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=7,
)


def evaluate(name: str, model) -> float:
    """Fit a regressor and return a compact test MAE summary."""
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    print(f"{name:28s} MAE = {mae:7.3f}")
    return mae

## 4. Linear and Regularized Regression

### Theory

Ordinary linear regression searches in the finite-dimensional affine class

$$
\mathcal F_{\mathrm{lin}}=\{x\mapsto x^\top\beta+\beta_0:\beta\in\mathbb R^p,\beta_0\in\mathbb R\}.
$$

With the intercept absorbed into the design matrix, the least-squares estimator solves

$$
\widehat\theta\in\arg\min_{\theta}\frac{1}{2N}\|y-X\theta\|_2^2.
$$

This is the basic regression contract: choose the affine function with smallest empirical
quadratic error. Regularized linear models keep the same prediction form but add a penalty that
controls complexity.

For Lasso,

$$
\widehat\beta\in\arg\min_\beta
\frac{1}{2N}\|y-X\beta\|_2^2+\lambda\|\beta\|_1.
$$

The $\ell_1$ penalty creates sparsity because its subgradient has a thresholding effect at zero.
ElasticNet uses

$$
\lambda\left(\alpha\|\beta\|_1+\frac{1-\alpha}{2}\|\beta\|_2^2\right),
$$

so it combines variable selection with ridge-type stabilization. Bayesian Ridge gives a
probabilistic interpretation to ridge shrinkage by assuming a Gaussian likelihood and a Gaussian
prior on coefficients. The posterior mode is a ridge estimator, and the posterior covariance gives
coefficient uncertainty.

`SGDRegressor` does not define a new function class by itself. It usually keeps a linear model but
fits it using stochastic gradient updates, which makes it appropriate when exact linear algebra is
too expensive.

In [2]:
from sklearn.linear_model import (
    BayesianRidge,
    ElasticNet,
    Lasso,
    LinearRegression,
    SGDRegressor,
)

linear_models = {
    "Linear regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Lasso": make_pipeline(StandardScaler(), Lasso(alpha=0.05, max_iter=10_000)),
    "ElasticNet": make_pipeline(
        StandardScaler(),
        ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=10_000),
    ),
    "SGDRegressor": make_pipeline(
        StandardScaler(),
        SGDRegressor(alpha=1e-4, max_iter=5_000, tol=1e-4, random_state=7),
    ),
    "BayesianRidge": make_pipeline(StandardScaler(), BayesianRidge()),
}

linear_mae = {name: evaluate(name, model) for name, model in linear_models.items()}

Linear regression            MAE =  10.523
Lasso                        MAE =  10.506
ElasticNet                   MAE =  10.044
SGDRegressor                 MAE =  10.511
BayesianRidge                MAE =  10.515


Interpretation. These models differ more in **regularization and optimization** than in the final
shape of the predictor. All of them still predict through an affine score in the input variables.
If their errors are similar, the simpler or more interpretable model is usually preferred. If Lasso
or ElasticNet performs competitively, the nonzero coefficients also provide a feature-selection
summary.

## 5. Kernel and Probabilistic Nonlinear Regression

### Theory

Kernel methods replace the explicit finite-dimensional vector $x$ by an implicit feature map
$\phi(x)$ into a Hilbert space $\mathcal H_k$. The inner product in that space is evaluated by a
positive-definite kernel:

$$
k(x,z)=\langle \phi(x),\phi(z)\rangle_{\mathcal H_k}.
$$

Kernel Ridge solves

$$
\widehat f\in\arg\min_{f\in\mathcal H_k}
\frac{1}{2N}\sum_{i=1}^N (y_i-f(x_i))^2+\frac{\lambda}{2}\|f\|_{\mathcal H_k}^2.
$$

By the representer theorem, the solution has the finite form

$$
\widehat f(\cdot)=\sum_{i=1}^N \widehat\alpha_i k(x_i,\cdot),
\qquad
\widehat\alpha=(K+N\lambda I)^{-1}y.
$$

SVR also uses a norm-controlled function in a feature space, but it replaces squared loss with an
$\epsilon$-insensitive loss. Errors inside the tube $|y_i-f(x_i)|\le\epsilon$ do not contribute to the
loss, so the fitted function is often determined by a subset of support vectors.

Gaussian Process Regression is the Bayesian version of kernel regression. Instead of selecting one
function directly, it assumes

$$
f\sim\mathcal{GP}(m,k),\qquad y_i=f(x_i)+\varepsilon_i,
\qquad \varepsilon_i\sim\mathcal N(0,\sigma^2).
$$

The kernel is now a covariance function. The output is not only a mean prediction but also a
predictive uncertainty.

In [3]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from sklearn.kernel_ridge import KernelRidge
from sklearn.svm import SVR

kernel_models = {
    "KernelRidge RBF": make_pipeline(
        StandardScaler(),
        KernelRidge(alpha=1.0, kernel="rbf", gamma=0.4),
    ),
    "SVR RBF": make_pipeline(
        StandardScaler(),
        SVR(kernel="rbf", C=10.0, epsilon=5.0, gamma="scale"),
    ),
    "Gaussian Process": make_pipeline(
        StandardScaler(),
        GaussianProcessRegressor(
            kernel=RBF(length_scale=1.0) + WhiteKernel(noise_level=1.0),
            alpha=1e-6,
            normalize_y=True,
            random_state=7,
        ),
    ),
}

kernel_mae = {name: evaluate(name, model) for name, model in kernel_models.items()}

KernelRidge RBF              MAE =  46.923
SVR RBF                      MAE =  62.988
Gaussian Process             MAE =  12.179


Interpretation. Kernel Ridge, SVR, and Gaussian processes are useful when the linear model is too
rigid. Their price is that training depends on pairwise sample comparisons. Exact kernel methods
therefore become expensive when the number of samples is large, because the Gram matrix has size
$N\times N$.

## 6. Tree-Based Ensembles

### Theory

Regression trees partition the input space into regions and predict a local average in each
region. A single tree is flexible but unstable: small changes in the data can change the splits.
Ensembles reduce this instability or bias by combining many trees.

A random forest averages independently randomized trees:

$$
\widehat f_{\mathrm{RF}}(x)=\frac{1}{B}\sum_{b=1}^B T_b(x).
$$

The randomness comes from bootstrap samples and random subsets of candidate features. This
reduces variance and makes the model robust on tabular data.

Gradient boosting builds an additive model

$$
F_M(x)=F_0(x)+\sum_{m=1}^M \eta h_m(x),
$$

where each new tree $h_m$ is fitted to improve the current model. In squared-error regression,
this is equivalent to fitting each new tree to residuals. XGBoost follows the same boosting idea
but uses a more optimized and regularized second-order objective, including shrinkage, tree
complexity penalties, column subsampling, and efficient split search.

In [4]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor

ensemble_models = {
    "RandomForest": RandomForestRegressor(
        n_estimators=120,
        max_depth=5,
        random_state=7,
        n_jobs=-1,
    ),
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=120,
        learning_rate=0.05,
        max_depth=3,
        random_state=7,
    ),
}

try:
    from xgboost import XGBRegressor
except ModuleNotFoundError:
    XGBRegressor = None

if XGBRegressor is not None:
    ensemble_models["XGBoost"] = XGBRegressor(
        n_estimators=120,
        learning_rate=0.05,
        max_depth=3,
        objective="reg:squarederror",
        random_state=7,
    )
else:
    print("XGBoost example skipped: xgboost is not installed in this environment.")

ensemble_mae = {name: evaluate(name, model) for name, model in ensemble_models.items()}

RandomForest                 MAE =  50.545
GradientBoosting             MAE =  33.703
XGBoost                      MAE =  34.248


Interpretation. Tree ensembles are often strong practical regressors because they capture
interactions and nonlinear thresholds without manually defining a feature map. Random forests
mainly reduce variance by averaging. Boosting mainly reduces bias by sequentially correcting the
current predictor. XGBoost is a production-grade version of boosted trees with additional
regularization and systems optimizations.

## 7. Compact Empirical Summary

The code below gathers the tiny examples into one table. The numbers are not the workshop answer
by themselves; they only illustrate the theoretical distinctions above on one controlled dataset.

In [5]:
all_mae = {}
all_mae.update(linear_mae)
all_mae.update(kernel_mae)
all_mae.update(ensemble_mae)

for name, mae in sorted(all_mae.items(), key=lambda item: item[1]):
    print(f"{name:28s} MAE = {mae:7.3f}")

ElasticNet                   MAE =  10.044
Lasso                        MAE =  10.506
SGDRegressor                 MAE =  10.511
BayesianRidge                MAE =  10.515
Linear regression            MAE =  10.523
Gaussian Process             MAE =  12.179
GradientBoosting             MAE =  33.703
XGBoost                      MAE =  34.248
KernelRidge RBF              MAE =  46.923
RandomForest                 MAE =  50.545
SVR RBF                      MAE =  62.988


## 8. Final Synthesis

The models differ along four main axes.

1. **Function class.** Linear, Lasso, ElasticNet, SGD, and Bayesian Ridge remain affine in the
original variables. Kernel Ridge, SVR, and Gaussian processes are nonlinear through a kernel.
Random forests and boosting are nonlinear through data-adaptive tree partitions.

2. **Regularization.** Lasso uses sparsity, ElasticNet mixes sparsity with shrinkage, Kernel Ridge
uses an RKHS norm, SVR uses margin control, Bayesian Ridge uses a Gaussian prior, and boosted
trees use shrinkage plus tree-complexity constraints.

3. **Optimization.** Ordinary least squares is solved by linear algebra. Lasso and ElasticNet are
usually solved by coordinate descent. SGD uses stochastic updates. Kernel Ridge and Gaussian
processes solve kernel linear systems. SVR solves a convex quadratic program. Forests and boosting
use greedy tree construction.

4. **Scalability.** Linear and stochastic models are the most scalable. Exact kernel and Gaussian
process methods are limited by the $N\times N$ Gram matrix. Tree ensembles are often the best
practical compromise for nonlinear tabular regression.

Therefore, the model choice should not be based only on predictive accuracy. It should also depend
on interpretability, sample size, feature dimension, nonlinear structure, uncertainty requirements,
and computational budget.

## 9. Interactive Empirical Analysis of Regression Assumptions

The final cells use a one-dimensional synthetic dataset to make the preceding theoretical
distinctions empirically visible. The purpose is not to replace the comparative tables, but
to show how model assumptions affect the fitted regression function: linear models produce
global trends, kernel models bend through similarity, and tree ensembles construct
piecewise-adaptive approximations.

In [6]:
import warnings

import matplotlib.pyplot as plt
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression
from sklearn.svm import SVR

rng_visual = np.random.default_rng(2026)
x_visual = np.linspace(-3.0, 3.0, 72)
y_visual = 0.8 * x_visual + 2.0 * np.sin(1.6 * x_visual) + rng_visual.normal(0.0, 0.35, x_visual.size)

X_visual = x_visual.reshape(-1, 1)
x_grid_visual = np.linspace(-3.3, 3.3, 360)
X_grid_visual = x_grid_visual.reshape(-1, 1)

def visual_mae(model) -> float:
    model.fit(X_visual, y_visual)
    return mean_absolute_error(y_visual, model.predict(X_visual))


def plot_curve(ax, title: str, model, color: str = "tab:blue") -> None:
    model.fit(X_visual, y_visual)
    y_hat = model.predict(X_grid_visual)
    mae = mean_absolute_error(y_visual, model.predict(X_visual))

    ax.scatter(x_visual, y_visual, s=18, color="black", alpha=0.65, label="sample")
    ax.plot(x_grid_visual, y_hat, color=color, lw=2.2, label="fit")
    ax.set_title(f"{title}\ntrain MAE = {mae:.2f}")
    ax.set_xlabel("x")
    ax.set_ylabel("prediction")
    ax.grid(alpha=0.18)

### Interactive Regression Model Explorer

The slider is a compact proxy for model flexibility. For regularized linear models it weakens the
penalty; for kernel methods it increases local flexibility; for trees it increases depth or the
number of boosting stages.

In [7]:
import ipywidgets as widgets
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel


def build_visual_model(model_name: str, complexity: float):
    complexity = float(complexity)

    if model_name == "LinearRegression":
        return make_pipeline(StandardScaler(), LinearRegression())

    if model_name == "Lasso":
        alpha = 10 ** (0.7 - 2.4 * complexity)
        return make_pipeline(StandardScaler(), Lasso(alpha=alpha, max_iter=10_000))

    if model_name == "ElasticNet":
        alpha = 10 ** (0.7 - 2.4 * complexity)
        return make_pipeline(
            StandardScaler(),
            ElasticNet(alpha=alpha, l1_ratio=0.45, max_iter=10_000),
        )

    if model_name == "KernelRidge":
        gamma = 0.05 + 2.8 * complexity
        alpha = 10 ** (0.2 - 1.8 * complexity)
        return make_pipeline(StandardScaler(), KernelRidge(alpha=alpha, kernel="rbf", gamma=gamma))

    if model_name == "SVR":
        c_value = 10 ** (0.3 + 1.7 * complexity)
        epsilon = 0.55 * (1.0 - complexity) + 0.03
        return make_pipeline(StandardScaler(), SVR(kernel="rbf", C=c_value, epsilon=epsilon))

    if model_name == "GaussianProcess":
        length_scale = 2.4 * (1.0 - complexity) + 0.18
        kernel = ConstantKernel(1.0, constant_value_bounds="fixed") * RBF(
            length_scale=length_scale,
            length_scale_bounds="fixed",
        ) + WhiteKernel(noise_level=0.12, noise_level_bounds="fixed")
        return make_pipeline(
            StandardScaler(),
            GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=7),
        )

    if model_name == "RandomForest":
        max_depth = 1 + int(round(7 * complexity))
        return RandomForestRegressor(
            n_estimators=80,
            max_depth=max_depth,
            min_samples_leaf=3,
            random_state=7,
            n_jobs=-1,
        )

    if model_name == "GradientBoosting":
        n_estimators = 8 + int(round(110 * complexity))
        return GradientBoostingRegressor(
            n_estimators=n_estimators,
            learning_rate=0.05,
            max_depth=2,
            random_state=7,
        )

    raise ValueError(f"Unknown model: {model_name}")


def draw_visual_model(model_name: str, complexity: float) -> None:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = build_visual_model(model_name, complexity)
        fig, ax = plt.subplots(figsize=(7.2, 3.8))
        plot_curve(ax, f"{model_name} | flexibility = {complexity:.2f}", model)
        ax.legend(loc="upper left", frameon=False)
        plt.show()

model_selector = widgets.Dropdown(
    options=[
        "LinearRegression",
        "Lasso",
        "ElasticNet",
        "KernelRidge",
        "SVR",
        "GaussianProcess",
        "RandomForest",
        "GradientBoosting",
    ],
    value="KernelRidge",
    description="model",
    layout=widgets.Layout(width="260px"),
)

complexity_slider = widgets.FloatSlider(
    value=0.55,
    min=0.0,
    max=1.0,
    step=0.05,
    description="flexibility",
    continuous_update=False,
    readout_format=".2f",
    layout=widgets.Layout(width="360px"),
)

interactive_view = widgets.interactive_output(
    draw_visual_model,
    {"model_name": model_selector, "complexity": complexity_slider},
)

display(widgets.VBox([widgets.HBox([model_selector, complexity_slider]), interactive_view]))

### Compact Animation of Model Flexibility

The animation compares four representative mechanisms. ElasticNet becomes less constrained,
Kernel Ridge becomes more local, Random Forest gets deeper, and Gradient Boosting receives more
stages.

In [8]:
def animate_flexibility_sweep(num_frames: int = 28) -> HTML:
    levels = np.linspace(0.0, 1.0, num_frames)

    fig, axes = plt.subplots(2, 2, figsize=(10.5, 6.2), sharex=True, sharey=True)
    axes = axes.ravel()
    specs = [
        ("ElasticNet", "tab:blue"),
        ("KernelRidge", "tab:purple"),
        ("RandomForest", "tab:green"),
        ("GradientBoosting", "tab:red"),
    ]

    lines = []
    labels = []
    for ax, (name, color) in zip(axes, specs):
        ax.scatter(x_visual, y_visual, s=13, color="black", alpha=0.45)
        line, = ax.plot([], [], color=color, lw=2.1)
        label = ax.text(0.03, 0.94, "", transform=ax.transAxes, va="top")
        ax.set_title(name)
        ax.grid(alpha=0.18)
        lines.append(line)
        labels.append(label)

    for ax in axes[2:]:
        ax.set_xlabel("x")
    for ax in axes[::2]:
        ax.set_ylabel("prediction")

    y_pad = 0.7
    y_min = min(y_visual.min(), -4.0) - y_pad
    y_max = max(y_visual.max(), 4.0) + y_pad
    for ax in axes:
        ax.set_xlim(x_grid_visual.min(), x_grid_visual.max())
        ax.set_ylim(y_min, y_max)

    def update(frame: int):
        level = float(levels[frame])
        artists = []
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            for line, label, (name, _) in zip(lines, labels, specs):
                model = build_visual_model(name, level)
                model.fit(X_visual, y_visual)
                prediction = model.predict(X_grid_visual)
                mae = mean_absolute_error(y_visual, model.predict(X_visual))
                line.set_data(x_grid_visual, prediction)
                label.set_text(f"flexibility={level:.2f}\nMAE={mae:.2f}")
                artists.extend([line, label])
        return artists

    animation = FuncAnimation(fig, update, frames=num_frames, interval=260, blit=False)
    plt.close(fig)
    return HTML(animation.to_jshtml())

animate_flexibility_sweep()